# Gravitino Access Control with Ranger

This notebook shows Gravitino enforcing **Ranger-backed authorization** on a Hive catalog. A single Spark session runs as user `lisa`, and her operations succeed or fail according to the role she currently holds. Changing her role from **developer** to **analyst** takes away her write privilege, and the same insert that worked before is then denied, enforced by Ranger, in the same session with no restart.

Start the playground with `./playground.sh start --enable-ranger`. Watch the policies at the Ranger admin UI, [http://localhost:6080](http://localhost:6080) (user `admin`, password `rangerR0cks!`).

In [ ]:
# Helper: print API responses, treating "already exists" as OK so the notebook
# is idempotent and can be re-run without alarming stack traces.
import json as _json

def show(resp, what=""):
    try:
        body = resp.json()
    except Exception:
        print(resp.status_code, resp.text[:300]); return
    code = body.get("code")
    typ = body.get("type", "")
    if code and code != 0:
        if "AlreadyExists" in typ:
            print(f"{what}: already exists (ok)")
        else:
            print(f"{what}: {typ} -> {body.get('message','')[:160]}")
    else:
        print(f"{what}: ok")

## Setup (as `manager`, via REST)

All setup runs through the Gravitino REST API as the `manager` user, so no admin Spark session is needed.

### Create the users

Register `manager` (the owner), `lisa` (our Spark user), and `bob` (a user who will stay unprivileged).

In [ ]:
import requests
import json

headers = {
    "Accept": "application/vnd.gravitino.v1+json",
    "Content-Type": "application/json",
}

for name in ["manager", "lisa"]:
    r = requests.post(
        "http://gravitino:8090/api/metalakes/metalake_demo/users",
        headers=headers, data=json.dumps({"name": name}))
    show(r, f"user {name}")

### Create a Hive catalog with Ranger authorization

The catalog is created with `authorization-provider = ranger`, so Gravitino pushes its role grants down into Ranger, which enforces them on Hive access.

In [ ]:
import requests
import json

url = "http://gravitino:8090/api/metalakes/metalake_demo/catalogs"
headers = {
    "Accept": "application/vnd.gravitino.v1+json",
    "Content-Type": "application/json",
    "Authorization": "Basic bWFuYWdlcjoxMjM=",  # manager:123
}
data = {
    "name": "catalog_hive_ranger",
    "type": "RELATIONAL",
    "provider": "hive",
    "comment": "comment",
    "properties": {
        "metastore.uris": "thrift://hive:9083",
        "authorization-provider": "ranger",
        "authorization.ranger.admin.url": "http://ranger:6080",
        "authorization.ranger.auth.type": "simple",
        "authorization.ranger.username": "admin",
        "authorization.ranger.password": "rangerR0cks!",
        "authorization.ranger.service.type": "HadoopSQL",
        "authorization.ranger.service.name": "hiveDev"
    }
}
response = requests.post(url, headers=headers, data=json.dumps(data))
show(response, "catalog catalog_hive_ranger")

## Start one Spark session as `lisa`

This is the **only** Spark session in the notebook, so no restart is ever required. Its user identity, `lisa`, is fixed here via `HADOOP_USER_NAME` and enforced by the Ranger Spark extension.

In [ ]:
import pyspark
import os
from pyspark.sql import SparkSession

os.environ["HADOOP_USER_NAME"] = "lisa"
gravitino_connector_jar = os.getenv("SPARK_CONNECTOR_JAR")

spark = SparkSession.builder \
    .appName("Access Control Demo") \
    .config("spark.plugins", "org.apache.gravitino.spark.connector.plugin.GravitinoSparkPlugin") \
    .config("spark.jars", f"/tmp/gravitino/packages/iceberg-spark-runtime-3.4_2.12-1.10.0.jar,\
                           /tmp/gravitino/packages/{gravitino_connector_jar},\
                           /tmp/gravitino/packages/kyuubi-spark-authz-shaded_2.12-1.9.2.jar") \
    .config("spark.sql.gravitino.uri", "http://gravitino:8090") \
    .config("spark.sql.gravitino.metalake", "metalake_demo") \
    .config("spark.sql.gravitino.enableIcebergSupport", "true") \
    .config("spark.sql.catalog.catalog_rest", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.catalog_rest.type", "rest") \
    .config("spark.sql.catalog.catalog_rest.uri", "http://gravitino:9001/iceberg/") \
    .config("spark.locality.wait.node", "0") \
    .config("spark.driver.extraClassPath", "/tmp/gravitino") \
    .config("spark.sql.extensions", "org.apache.kyuubi.plugin.spark.authz.ranger.RangerSparkExtension") \
    .config("spark.sql.warehouse.dir", "hdfs://hive:9000/user/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark session started as:", os.environ["HADOOP_USER_NAME"])

## `lisa` as a Developer

Grant `lisa` a **developer** role (use, create, modify, select on the `access_control` schema), then show she can create a table, insert rows, and read them back, all allowed.

In [ ]:
import requests, json

url = "http://gravitino:8090/api/metalakes/metalake_demo/roles"
headers = {"Accept": "application/vnd.gravitino.v1+json", "Content-Type": "application/json"}
data = {
    "name": "developer",
    "properties": {"k1": "v1"},
    "securableObjects": [
        {"fullName": "catalog_hive_ranger", "type": "CATALOG",
         "privileges": [{"name": "USE_CATALOG", "condition": "ALLOW"}]},
        {"fullName": "catalog_hive_ranger.access_control", "type": "SCHEMA",
         "privileges": [
            {"name": "USE_SCHEMA", "condition": "ALLOW"},
            {"name": "CREATE_TABLE", "condition": "ALLOW"},
            {"name": "MODIFY_TABLE", "condition": "ALLOW"},
            {"name": "SELECT_TABLE", "condition": "ALLOW"}]}
    ]
}
show(requests.post(url, headers=headers, data=json.dumps(data)), "role developer")

grant = "http://gravitino:8090/api/metalakes/metalake_demo/permissions/users/lisa/grant"
show(requests.put(grant, headers=headers, data=json.dumps({"roleNames": ["developer"]})), "grant developer to lisa")

### Create a table and add data (allowed)

In [ ]:
# Fully-qualified names so nothing depends on the current catalog/schema.
spark.sql("CREATE SCHEMA IF NOT EXISTS catalog_hive_ranger.access_control")
spark.sql("""CREATE TABLE IF NOT EXISTS catalog_hive_ranger.access_control.customers
             (customer_id int, customer_name string, customer_email string)""")
spark.sql("""INSERT INTO catalog_hive_ranger.access_control.customers
             VALUES (11,'Rory Brown','rory@123.com'), (12,'Jerry Washington','jerry@dt.com')""")
spark.sql("SELECT * FROM catalog_hive_ranger.access_control.customers").show()

### Transfer table ownership to `manager`

`lisa` created the table, so she is its **owner**, and in Gravitino an owner can act on the object regardless of role (discretionary access control). To make the role change below actually bite, transfer ownership to `manager`. Now `lisa`'s access depends purely on the role she holds.

In [ ]:
import requests, json

# Set the owner of the customers table to manager (was lisa, who created it).
url = "http://gravitino:8090/api/metalakes/metalake_demo/owners/table/catalog_hive_ranger.access_control.customers"
headers = {"Accept": "application/vnd.gravitino.v1+json", "Content-Type": "application/json"}
r = requests.put(url, headers=headers, data=json.dumps({"name": "manager", "type": "USER"}))
show(r, "set customers owner to manager")

# Confirm the new owner
who = requests.get(url, headers=headers)
print("current owner:", who.text)

## Change `lisa` to an Analyst

Revoke the developer role and grant an **analyst** role, which allows select but not modify. In the same Spark session, `lisa` can now read the table but her insert is denied by Ranger.

In [ ]:
import requests, json
headers = {"Accept": "application/vnd.gravitino.v1+json", "Content-Type": "application/json"}

revoke = "http://gravitino:8090/api/metalakes/metalake_demo/permissions/users/lisa/revoke"
show(requests.put(revoke, headers=headers, data=json.dumps({"roleNames": ["developer"]})), "revoke developer from lisa")

url = "http://gravitino:8090/api/metalakes/metalake_demo/roles"
data = {
    "name": "analyst",
    "properties": {"k1": "v1"},
    "securableObjects": [
        {"fullName": "catalog_hive_ranger", "type": "CATALOG",
         "privileges": [{"name": "USE_CATALOG", "condition": "ALLOW"}]},
        {"fullName": "catalog_hive_ranger.access_control", "type": "SCHEMA",
         "privileges": [
            {"name": "USE_SCHEMA", "condition": "ALLOW"},
            {"name": "SELECT_TABLE", "condition": "ALLOW"}]}
    ]
}
show(requests.post(url, headers=headers, data=json.dumps(data)), "role analyst")

grant = "http://gravitino:8090/api/metalakes/metalake_demo/permissions/users/lisa/grant"
show(requests.put(grant, headers=headers, data=json.dumps({"roleNames": ["analyst"]})), "grant analyst to lisa")

### Allowed: select

In [ ]:
spark.sql("SELECT * FROM catalog_hive_ranger.access_control.customers").show()

### Denied: insert (analyst has no MODIFY_TABLE)

In [ ]:
from py4j.protocol import Py4JJavaError
try:
    spark.sql("INSERT INTO catalog_hive_ranger.access_control.customers VALUES (13,'New Person','new@dt.com')")
    print("Unexpected: insert succeeded")
except Py4JJavaError as e:
    print("Denied as expected:")
    print(e.java_exception)

---

The same insert that succeeded under the developer role was denied under the analyst role, enforced by Ranger on `lisa`'s live Spark session with no kernel restart. The policies are visible at [http://localhost:6080](http://localhost:6080). See the [access control documentation](https://gravitino.apache.org/docs/latest/security/access-control) for details.